# Workflow 1: Identify Native Non-Covalent Lasso Entanglements (NCLEs)

## Goal

Identify all native NCLEs present in a reference protein structure, filter for high-confidence entanglements, cluster redundant variants, and compute structural features for the representative set.

---

## Typical runtime

| Step | Runtime |
|------|---------|
| Native entanglement detection | 1–10 minutes |
| High-quality filtering | < 1 minute |
| Clustering | < 1 minute |
| Feature generation | 1–5 minutes |

---

## Required input files

| File | Path | Role |
|------|------|------|
| Cleaned all-atom PDB | `$DATASTORE/user_input/reference_structures/1zmr_model_clean.pdb` | Input structure |
| Cα PSF topology | `$DATASTORE/user_input/reference_structures/1zmr_model_clean_ca.psf` | Topology reference |
| Cα COR coordinates | `$DATASTORE/user_input/reference_structures/1zmr_model_clean_ca.cor` | Coordinate reference |

> **Note:** This tutorial demonstrates analysis on an all-atom structure, but EntDetect can also handle Cα coarse-grained structures.

---

> **Environment:** Before running this notebook, activate the `entdetect` conda environment and register it as a kernel:
> ```bash
> conda activate entdetect
> python -m ipykernel install --user --name entdetect --display-name "entdetect"
> ```
> Then select **entdetect** from the kernel picker (top-right of VS Code or Jupyter).

## Step 1. Set up paths

Set the DATASTORE root and create all output directories needed for this workflow.  
*(The `conda activate` command is handled outside the notebook — see the environment note above.)*

In [27]:
import os

DATASTORE = "/scratch/ims86/EntDetect_Datastore"
OUTDIR = f"{DATASTORE}/outputs/notebook_tmp/workflow1"
NCLE_outdir = f"{OUTDIR}/NCLE"
HQ_NCLE_outdir = f"{OUTDIR}/HQ_NCLE"
clustered_HQ_NCLE_outdir = f"{OUTDIR}/clustered_HQ_NCLE"
clustered_HQ_NCLE_feature_outdir = f"{OUTDIR}/clustered_HQ_NCLE_features"

os.makedirs(NCLE_outdir,                        exist_ok=True)
os.makedirs(HQ_NCLE_outdir,                    exist_ok=True)
os.makedirs(clustered_HQ_NCLE_outdir,          exist_ok=True)
os.makedirs(clustered_HQ_NCLE_feature_outdir, exist_ok=True)

print(f"DATASTORE : {DATASTORE}")
print(f"OUTDIR    : {OUTDIR}")
print("Output directories ready.")

DATASTORE : /scratch/ims86/EntDetect_Datastore
OUTDIR    : /scratch/ims86/EntDetect_Datastore/outputs/notebook_tmp/workflow1
Output directories ready.


## Step 2. Prepare a cleaned structure

The `1zmr_model_clean.pdb` file in `$DATASTORE/user_input/reference_structures/` is already cleaned and ready to use. For your own protein you would:

- rebuild missing residues and atoms (e.g., with **Modeller** or **CHARMM**);
- remove duplicate residue records;
- verify chain labels and residue numbering are sensible;
- ensure the PDB contains only the protein chain(s) of interest.

**Output:** `$OUTDIR/NCLE/1ZMR_GE.csv`

In [28]:
from EntDetect.gaussian_entanglement import GaussianEntanglement

# ── Inputs ──────────────────────────────────────────────────────────────────
PDB = f"{DATASTORE}/user_input/reference_structures/1zmr_model_clean.pdb"
CHAIN = "A"
ID = "1ZMR"

# ── Initialize ───────────────────────────────────────────────────────────────
ge = GaussianEntanglement(g_threshold=0.6, density=0.0, Calpha=False, CG=False,
                          ent_detection_method=2)

# ── Run ──────────────────────────────────────────────────────────────────────
ge.calculate_native_entanglements(pdb_file=PDB, outdir=NCLE_outdir, ID=ID, chain=CHAIN)

2026-06-24 20:48:13 [EntDetect.GaussianEntanglement] INFO: Creating directory: /scratch/ims86/EntDetect_Datastore/outputs/notebook_tmp/workflow1/NCLE/unmapped_missing_residues
2026-06-24 20:48:13 [EntDetect.GaussianEntanglement] INFO: 
####################################################################################################
COMPUTING ENTANGLEMENTS FOR 1zmr_model_clean with ID 1ZMR
2026-06-24 20:48:13 [EntDetect.GaussianEntanglement] INFO: Checking for disulfide bonds
2026-06-24 20:48:13 [EntDetect.GaussianEntanglement] INFO: All-atom model and contacts: Selecting all heavy atoms (no hydrogens) for entanglement calculations
2026-06-24 20:48:24 [EntDetect.GaussianEntanglement] INFO: SAVED: /scratch/ims86/EntDetect_Datastore/outputs/notebook_tmp/workflow1/NCLE/1ZMR_GE.csv


{'outfile': '/scratch/ims86/EntDetect_Datastore/outputs/notebook_tmp/workflow1/NCLE/1ZMR_GE.csv',
 'ent_result':        ID chain    i    j crossingsN crossingsC       gn       gc  GLNn  GLNc  \
 0    1ZMR     A    1  171        NaN        NaN  0.00000 -0.07654     0     0   
 1    1ZMR     A    1  179        NaN        NaN  0.00000  0.09663     0     0   
 2    1ZMR     A    1  369        NaN        NaN  0.00000 -0.04075     0     0   
 3    1ZMR     A    1  378        NaN        NaN  0.00000  0.00000     0     0   
 4    1ZMR     A    1  382        NaN        NaN  0.00000  0.00000     0     0   
 5    1ZMR     A    2  381        NaN        NaN  0.00000  0.00000     0     0   
 6    1ZMR     A    2  382        NaN        NaN  0.00000  0.00000     0     0   
 7    1ZMR     A    2  385        NaN        NaN  0.00000  0.00000     0     0   
 8    1ZMR     A    3  163        NaN        NaN  0.00000 -0.10159     0     0   
 9    1ZMR     A    3  171        NaN        NaN  0.00000  0.02031  

In [29]:
import pandas as pd

result_step3 = pd.read_csv(f"{native_outdir}/1ZMR_GE.csv", sep='|')
print(f"Detected entanglements: {len(result_step3)}")
result_step3.head()

Detected entanglements: 833


,ID,chain,i,j,crossingsN,crossingsC,gn,gc,GLNn,GLNc,TLNn,TLNc,CCbond,ENT
0,1ZMR,A,1,171,NaN,NaN,0.0,-0.07654,0,0,0,0,False,False
1,1ZMR,A,1,179,NaN,NaN,0.0,0.09663,0,0,0,0,False,False
2,1ZMR,A,1,369,NaN,NaN,0.0,-0.04075,0,0,0,0,False,False
3,1ZMR,A,1,378,NaN,NaN,0.0,0.00000,0,0,0,0,False,False
4,1ZMR,A,1,382,NaN,NaN,0.0,0.00000,0,0,0,0,False,False


**Output:** `$OUTDIR/HQ_NCLE/1ZMR.csv`

In [30]:
from EntDetect.gaussian_entanglement import GaussianEntanglement

PDB = f"{DATASTORE}/user_input/reference_structures/1zmr_model_clean.pdb"
ID = "1ZMR"

# Point to Step 3 output
NCLE_file = f"{NCLE_outdir}/1ZMR_GE.csv"

ge = GaussianEntanglement(g_threshold=0.6, density=0.0, Calpha=False, CG=False,
                          ent_detection_method=2)

ge.select_high_quality_entanglements(
    NCLE_file,
    PDB,
    outdir=HQ_NCLE_outdir,
    ID=ID,
    model="EXP",   # "EXP" for experimental structures; "AF" for AlphaFold models
    chain=CHAIN,
)

2026-06-24 20:48:24 [EntDetect.GaussianEntanglement] INFO: GE FILE: /scratch/ims86/EntDetect_Datastore/outputs/notebook_tmp/workflow1/NCLE/1ZMR_GE.csv
2026-06-24 20:48:24 [EntDetect.GaussianEntanglement] INFO: 
##################################################
Removing slipknots...
2026-06-24 20:48:24 [EntDetect.GaussianEntanglement] INFO: SAVED: /scratch/ims86/EntDetect_Datastore/outputs/notebook_tmp/workflow1/HQ_NCLE/1ZMR.csv


{'outfile': '/scratch/ims86/EntDetect_Datastore/outputs/notebook_tmp/workflow1/HQ_NCLE/1ZMR.csv',
 'GE_data':       ID chain    i    j  crossingsN  crossingsC       gn       gc  GLNn  \
 0   1ZMR     A   15   99         NaN       108.0 -0.03377  1.01395     0   
 1   1ZMR     A   15  101         NaN       108.0 -0.02786  0.91393     0   
 2   1ZMR     A   39  166        18.0         NaN  0.86554  0.01968     1   
 3   1ZMR     A   40  166        18.0         NaN  0.81444 -0.00027     1   
 4   1ZMR     A   43  139        17.0         NaN  0.79379  0.30363     1   
 5   1ZMR     A   43  165        18.0         NaN  0.83725 -0.03072     1   
 6   1ZMR     A   43  166        18.0         NaN  0.81764 -0.01722     1   
 7   1ZMR     A   46  165        17.0         NaN  0.76778 -0.02968     1   
 8   1ZMR     A   52  102         NaN       108.0  0.26638  0.61363     0   
 9   1ZMR     A   53  101         NaN       108.0  0.20966  0.71005     0   
 10  1ZMR     A   53  102         NaN       

In [31]:
result_step4 = pd.read_csv(f"{native_HQ_outdir}/1ZMR.csv", sep='|')
print(f"High-quality entanglements: {len(result_step4)}")
result_step4.head()

High-quality entanglements: 54


,ID,chain,i,j,crossingsN,crossingsC,gn,gc,GLNn,GLNc,TLNn,TLNc,CCbond,ENT,Slipknot_N,Slipknot_C
0,1ZMR,A,15,99,NaN,108.0,-0.03377,1.01395,0,1,0,1,False,True,False,False
1,1ZMR,A,15,101,NaN,108.0,-0.02786,0.91393,0,1,0,1,False,True,False,False
2,1ZMR,A,39,166,18.0,NaN,0.86554,0.01968,1,0,1,0,False,True,False,False
3,1ZMR,A,40,166,18.0,NaN,0.81444,-0.00027,1,0,1,0,False,True,False,False
4,1ZMR,A,43,139,17.0,NaN,0.79379,0.30363,1,0,1,0,False,True,False,False


**Output:** `$OUTDIR/clustered_HQ_NCLE/1ZMR.csv`

In [32]:
from EntDetect.clustering import ClusterNativeEntanglements

outfile = "1ZMR.csv"

# Point to Step 4 output
NCLE_file = f"{HQ_NCLE_outdir}/1ZMR.csv"

clustering = ClusterNativeEntanglements(organism="Ecoli", cut_off=None)
clustering.Cluster_NativeEntanglements(
    NCLE_file,
    outdir=clustered_HQ_NCLE_outdir,
    outfile=outfile,
    chain=CHAIN,
)

2026-06-24 20:48:24 [EntDetect.ClusterNativeEntanglements] INFO: Clustering Ecoli Native Entanglements with dist_cutoff: 57
2026-06-24 20:48:24 [EntDetect.ClusterNativeEntanglements] INFO: Loading /scratch/ims86/EntDetect_Datastore/outputs/notebook_tmp/workflow1/HQ_NCLE/1ZMR.csv
2026-06-24 20:48:24 [EntDetect.ClusterNativeEntanglements] INFO: # Step 1
2026-06-24 20:48:24 [EntDetect.ClusterNativeEntanglements] INFO: 
2026-06-24 20:48:24 [EntDetect.ClusterNativeEntanglements] INFO: STEP 1 SUMMARY: Data Loading and Grouping by Unique Crossing Sets
2026-06-24 20:48:24 [EntDetect.ClusterNativeEntanglements] INFO: ====================================================================================================
2026-06-24 20:48:24 [EntDetect.ClusterNativeEntanglements] INFO: Total raw entanglements loaded: 54
2026-06-24 20:48:24 [EntDetect.ClusterNativeEntanglements] INFO: Number of protein IDs: 1
2026-06-24 20:48:24 [EntDetect.ClusterNativeEntanglements] INFO:   - 1ZMR: 54 raw entanglemen

{'outfile': '/scratch/ims86/EntDetect_Datastore/outputs/notebook_tmp/workflow1/clustered_HQ_NCLE/1ZMR.csv',
 'ent_result':      ID chain    i    j  crossingsN  crossingsC       gn       gc  GLNn  GLNc  \
 0  1ZMR     A  228  274       221.0         NaN  0.63806 -0.07150     1     0   
 1  1ZMR     A   55   99         NaN       108.0  0.15095  0.66113     0     1   
 2  1ZMR     A   43  139        17.0         NaN  0.79379  0.30363     1     0   
 3  1ZMR     A  271  287      -257.0         NaN -0.86397  0.05277    -1     0   
 
    TLNn  TLNc  num_contacts  \
 0     1     0            29   
 1     0     1             8   
 2     1     0             6   
 3    -1     0            11   
 
                                             contacts  CCBond  
 0  227-274;227-275;228-274;228-277;229-277;230-27...   False  
 1  15-99;15-101;52-102;53-101;53-102;54-101;55-99...   False  
 2          43-139;46-165;39-166;40-166;43-165;43-166   False  
 3  271-287;271-288;264-315;265-315;266-317;267-

In [33]:
result_step5 = pd.read_csv(f"{native_clustered_HQ_outdir}/1ZMR.csv", sep='|')
print(f"Representative entanglements: {len(result_step5)}")
result_step5.head()

Representative entanglements: 4


,ID,chain,i,j,crossingsN,crossingsC,gn,gc,GLNn,GLNc,TLNn,TLNc,num_contacts,contacts,CCBond
0,1ZMR,A,228,274,221.0,NaN,0.63806,-0.07150,1,0,1,0,29,227-274;227-275;228-274;228-277;229-277;230-27...,False
1,1ZMR,A,55,99,NaN,108.0,0.15095,0.66113,0,1,0,1,8,15-99;15-101;52-102;53-101;53-102;54-101;55-99...,False
2,1ZMR,A,43,139,17.0,NaN,0.79379,0.30363,1,0,1,0,6,43-139;46-165;39-166;40-166;43-165;43-166,False
3,1ZMR,A,271,287,-257.0,NaN,-0.86397,0.05277,-1,0,-1,0,11,271-287;271-288;264-315;265-315;266-317;267-31...,False


**Output:** `$OUTDIR/clustered_HQ_NCLE_features/P00558_1ZMR_A_uent_features.csv`

In [34]:
from EntDetect.entanglement_features import FeatureGen

PDB = f"{DATASTORE}/user_input/reference_structures/1zmr_model_clean.pdb"
pdb = PDB
UNIPROT = "P00558"
PDBID = "1ZMR"

# Point to Step 5 output
cluster_file = f"{clustered_HQ_NCLE_outdir}/1ZMR.csv"

FGen = FeatureGen(PDB, outdir=clustered_HQ_NCLE_feature_outdir, cluster_file=cluster_file)

# gene  : UniProt accession for ecPGK
# chain : PDB chain identifier
# pdbid : four-letter PDB code (uppercase)
EntFeatures = FGen.get_uent_features(gene=UNIPROT, chain=CHAIN, pdbid=PDBID)
print(EntFeatures)

2026-06-24 20:48:24 [EntDetect.FeatureGen] INFO: Unique entanglement features saved to /scratch/ims86/EntDetect_Datastore/outputs/notebook_tmp/workflow1/clustered_HQ_NCLE_features/P00558_1ZMR_A_uent_features.csv


{'outfile': '/scratch/ims86/EntDetect_Datastore/outputs/notebook_tmp/workflow1/clustered_HQ_NCLE_features/P00558_1ZMR_A_uent_features.csv', 'results':      gene   PDB chain  ENT-ID       gn  N_term_thread       gc  C_term_thread  \
0  P00558  1ZMR     A       0  0.63806              1 -0.07150              0   
1  P00558  1ZMR     A       1  0.15095              0  0.66113              1   
2  P00558  1ZMR     A       2  0.79379              1  0.30363              0   
3  P00558  1ZMR     A       3 -0.86397              1  0.05277              0   

     i    j  ... min_N_prot_depth_left min_N_thread_depth_left  \
0  228  274  ...              0.529976                0.969298   
1   55   99  ...                   NaN                     NaN   
2   43  139  ...              0.040767                0.395349   
3  271  287  ...              0.616307                0.948339   

  min_N_thread_slippage_left min_C_prot_depth_right min_C_thread_depth_right  \
0                      221.0    

In [35]:
print(f'Loading feature data from {clustered_HQ_NCLE_feature_outdir}/P00558_1ZMR_A_uent_features.csv')
result_step6 = pd.read_csv(f"{clustered_HQ_NCLE_feature_outdir}/P00558_1ZMR_A_uent_features.csv", sep='|')
print(f"Feature rows: {len(result_step6)}")
result_step6.head()

Loading feature data from /scratch/ims86/EntDetect_Datastore/outputs/notebook_tmp/workflow1/clustered_HQ_NCLE_features/P00558_1ZMR_A_uent_features.csv
Feature rows: 4


,gene,PDB,chain,ENT-ID,gn,N_term_thread,gc,C_term_thread,i,j,...,min_N_prot_depth_left,min_N_thread_depth_left,min_N_thread_slippage_left,min_C_prot_depth_right,min_C_thread_depth_right,min_C_thread_slippage_right,prot_size,ACO,RCO,CCBond
0,P00558,1ZMR,A,0,0.63806,1,-0.07150,0,228,274,...,0.529976,0.969298,221.0,NaN,NaN,NaN,417,89.413793,0.214422,False
1,P00558,1ZMR,A,1,0.15095,0,0.66113,1,55,99,...,NaN,NaN,NaN,0.741007,0.971698,309.0,417,56.750000,0.136091,False
2,P00558,1ZMR,A,2,0.79379,1,0.30363,0,43,139,...,0.040767,0.395349,17.0,NaN,NaN,NaN,417,118.833333,0.284972,False
3,P00558,1ZMR,A,3,-0.86397,1,0.05277,0,271,287,...,0.616307,0.948339,257.0,NaN,NaN,NaN,417,43.818182,0.105080,False


## Step 7. Visualize the representative entanglements

The interactive viewer below renders the 1ZMR structure directly in the notebook using **py3Dmol**.

| Colour | Meaning |
|--------|---------|
| Light grey | Rest of the chain |
| **Red** | Loop residues (`i` → `j`) for each representative NCLE |
| **Blue** + sphere | Crossing residues (`crossingsN` / `crossingsC`) |

For publication-quality figures, load the representative structures in **VMD** or **PyMOL** using the residue ranges identified here.

In [36]:
import py3Dmol
import pandas as pd
import json

# Load representative NCLEs
clustered_ncles = pd.read_csv(f"{clustered_HQ_NCLE_outdir}/1ZMR.csv", sep='|').reset_index(drop=True)

ncles = []
for idx, row in clustered_ncles.iterrows():
    loop_start = int(row['i'])
    loop_end = int(row['j'])
    loop_sel = f"{loop_start}-{loop_end}"

    crossing_n = str(row['crossingsN']).strip()
    crossing_c = str(row['crossingsC']).strip()
    crossings = []
    if crossing_n.lower() != 'nan' and crossing_n:
        crossings.append(str(int(float(crossing_n.lstrip('+-')))))
    if crossing_c.lower() != 'nan' and crossing_c:
        crossings.append(str(int(float(crossing_c.lstrip('+-')))))

    cross_parts = []
    if crossings:
        if crossing_n.lower() != 'nan' and crossing_n:
            cross_parts.append(f"N-cross:{crossings[0]}")
        if crossing_c.lower() != 'nan' and crossing_c:
            cross_parts.append(f"C-cross:{crossings[-1]}")
    cross_label = ' | '.join(cross_parts) if cross_parts else 'no crossing'

    ncles.append({
        'label': f"NCLE {idx+1}: loop {loop_start}-{loop_end} | {cross_label}",
        'loop_start': loop_start,
        'loop_end': loop_end,
        'loop_sel': loop_sel,
        'cross_sel': ','.join(crossings),
    })

first_ncle = ncles[0]
print(f"loop_sel: {first_ncle['loop_sel']}")
print(f"crossing_sel: {first_ncle['cross_sel'] or 'none'}")

with open(PDB) as f:
    pdb_str = f.read()

# Working render path: full-structure zoom + first NCLE loop in red + crossing in blue.
view = py3Dmol.view(width=820, height=520)
view.addModel(pdb_str, 'pdb')
view.setStyle({'chain': 'A'}, {'cartoon': {'color': '#bfbfbf', 'opacity': 0.9}})
view.setStyle({'chain': 'A', 'resi': first_ncle['loop_sel']}, {'cartoon': {'color': '#E57373'}})
if first_ncle['cross_sel']:
    view.setStyle(
        {'chain': 'A', 'resi': first_ncle['cross_sel']},
        {'cartoon': {'color': '#1E88E5'}, 'sphere': {'color': '#1E88E5', 'radius': 0.7}},
)
view.zoomTo({'chain': 'A'})
view.show()

# Reused by the dropdown controller cell below.
vname = f"viewer_{view.uniqueid}"
ncles_json = json.dumps(ncles)

print(
    f"Rendered base 1ZMR structure with first NCLE loop highlighted in red: "
    f"{first_ncle['loop_start']}-{first_ncle['loop_end']}"
)
print(f"Highlighted crossing residue in blue: {first_ncle['cross_sel'] or 'none'}")



loop_sel: 228-274
crossing_sel: 221


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Rendered base 1ZMR structure with first NCLE loop highlighted in red: 228-274
Highlighted crossing residue in blue: 221


## I/O Reference for run_nativeNCLE.py

Each file below is listed once, followed by the column-level description table for that file.

### `$DATASTORE/user_input/reference_structures/1zmr_model_clean.pdb`

| I/O | File | File Description |
|---|---|---|
| Input | `$DATASTORE/user_input/reference_structures/1zmr_model_clean.pdb` | Cleaned all-atom structure used as the geometric source for NCLE discovery. |

### `$OUTDIR/NCLE/1ZMR_GE.csv`

| Column Name | Column Description |
|---|---|
| ID | Protein or structure identifier for the entanglement record. |
| chain | Chain identifier in the input structure. |
| i | First residue index of the native contact that defines the loop start. |
| j | Second residue index of the native contact that defines the loop end. |
| crossingsN | Comma-separated N-terminal crossing residues, stored with chirality signs such as +109 or -258. |
| crossingsC | Comma-separated C-terminal crossing residues, stored with chirality signs such as +109 or -258. |
| gn | Raw Gaussian entanglement score for the N-terminal side of the loop. |
| gc | Raw Gaussian entanglement score for the C-terminal side of the loop. |
| GLNn | Rounded Gaussian linking number for the N-terminal side, derived from gn. |
| GLNc | Rounded Gaussian linking number for the C-terminal side, derived from gc. |
| TLNn | Topological linking number for the N-terminal side after Topoly-based crossing assignment and buffer filtering. |
| TLNc | Topological linking number for the C-terminal side after Topoly-based crossing assignment and buffer filtering. |
| CCbond | Boolean flag indicating that the loop contact is a covalent C-C bond, typically a disulfide-linked loop. |
| ENT | Boolean flag showing whether the native contact passed the chosen entanglement detection criterion. |

### `$OUTDIR/HQ_NCLE/1ZMR.csv`

| Column Name | Column Description |
|---|---|
| ID | Protein or structure identifier for the filtered entanglement record. |
| chain | Chain identifier in the input structure. |
| i | First residue index of the loop-defining contact. |
| j | Second residue index of the loop-defining contact. |
| crossingsN | N-terminal crossing residues retained after slipknot filtering. |
| crossingsC | C-terminal crossing residues retained after slipknot filtering. |
| gn | Raw Gaussian entanglement score for the N-terminal side. |
| gc | Raw Gaussian entanglement score for the C-terminal side. |
| GLNn | Rounded Gaussian linking number for the N-terminal side. |
| GLNc | Rounded Gaussian linking number for the C-terminal side. |
| TLNn | Topological linking number for the N-terminal side. |
| TLNc | Topological linking number for the C-terminal side. |
| CCbond | Boolean flag indicating a covalent C-C loop closure. |
| ENT | Boolean flag showing whether the record is still classified as an entanglement after quality filtering. |
| Slipknot_N | Boolean flag indicating that the N-terminal crossing pattern is consistent with a slipknot and should be treated cautiously. |
| Slipknot_C | Boolean flag indicating that the C-terminal crossing pattern is consistent with a slipknot and should be treated cautiously. |

### `$OUTDIR/clustered_HQ_NCLE/1ZMR.csv`

| Column Name | Column Description |
|---|---|
| ID | Protein or structure identifier for the clustered representative. |
| chain | Chain identifier in the input structure. |
| i | First residue index of the representative loop contact. |
| j | Second residue index of the representative loop contact. |
| crossingsN | Representative N-terminal crossing residues for the cluster. |
| crossingsC | Representative C-terminal crossing residues for the cluster. |
| gn | Representative raw Gaussian entanglement score for the N-terminal side. |
| gc | Representative raw Gaussian entanglement score for the C-terminal side. |
| GLNn | Representative rounded Gaussian linking number for the N-terminal side. |
| GLNc | Representative rounded Gaussian linking number for the C-terminal side. |
| TLNn | Representative topological linking number for the N-terminal side. |
| TLNc | Representative topological linking number for the C-terminal side. |
| num_contacts | Number of raw entanglement records merged into the representative cluster. |
| contacts | Semicolon-separated list of the loop residue pairs that were merged into the representative cluster. |
| CCBond | Boolean flag indicating whether the representative loop closure is a covalent C-C bond. |

### `$OUTDIR/clustered_HQ_NCLE_features/P00558_1ZMR_A_uent_features.csv`

| Column Name | Column Description |
|---|---|
| gene | UniProt accession used for protein-level annotation and output naming. |
| PDB | Four-letter PDB identifier for the structure. |
| chain | Chain identifier used for feature extraction. |
| ENT-ID | Row index of the representative entanglement within the clustered input file. |
| gn | Representative raw Gaussian entanglement score for the N-terminal side. |
| N_term_thread | Number of N-terminal crossing residues threading through the loop. |
| gc | Representative raw Gaussian entanglement score for the C-terminal side. |
| C_term_thread | Number of C-terminal crossing residues threading through the loop. |
| i | First residue index of the representative loop contact. |
| j | Second residue index of the representative loop contact. |
| NC | Core loop contact residues, stored as the pair i,j. |
| NC_wbuff | Loop residues expanded by the fixed residue buffer used for neighborhood calculations. |
| NC_region | Residues whose side-chain heavy atoms lie within the neighborhood cutoff of the buffered loop residues. |
| crossingsN | Core N-terminal crossing residues, without the buffer expansion. |
| crossingsC | Core C-terminal crossing residues, without the buffer expansion. |
| crossingsN_wbuff | N-terminal crossing residues expanded by the fixed residue buffer. |
| crossingsC_wbuff | C-terminal crossing residues expanded by the fixed residue buffer. |
| crossingsN_region | Residues whose side-chain heavy atoms lie within the neighborhood cutoff of the buffered N-terminal crossings. |
| crossingsC_region | Residues whose side-chain heavy atoms lie within the neighborhood cutoff of the buffered C-terminal crossings. |
| ent_region | Union of the loop neighborhood, crossing neighborhoods, core loop residues, and core crossing residues. |
| loopsize | Loop length in residues, computed as j minus i. |
| num_zipper_nc | Number of raw entanglement contacts that were merged into the representative cluster. |
| perc_bb_loop | Loop length normalized by the full protein length. |
| num_loop_contacting_res | Number of residues whose side-chain heavy atoms contact the loop neighborhood. |
| num_cross_nearest_neighbors | Total number of residues near the N- and C-terminal crossing neighborhoods. |
| ent_coverage | Fraction of the protein covered by the full entangled region. |
| min_N_prot_depth_left | Fractional distance of the closest N-terminal crossing from the protein N-terminus. |
| min_N_thread_depth_left | Fractional distance of the closest N-terminal crossing from the loop start. |
| min_N_thread_slippage_left | Residue offset of the closest N-terminal crossing from the loop start. |
| min_C_prot_depth_right | Fractional distance of the closest C-terminal crossing from the protein C-terminus. |
| min_C_thread_depth_right | Fractional distance of the closest C-terminal crossing from the loop end. |
| min_C_thread_slippage_right | Residue offset of the closest C-terminal crossing from the protein C-terminus. |
| prot_size | Full protein length from the UniProt canonical sequence. |
| ACO | Average contact order of the merged loop contacts in the representative cluster. |
| RCO | Relative contact order, computed as ACO divided by prot_size. |
| CCBond | Boolean flag indicating whether the representative loop closure is a covalent C-C bond. |
